In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/ufc-rag-proj')
print("current working directory", os.getcwd())


Mounted at /content/drive
current working directory /content/drive/MyDrive/ufc-rag-proj


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import missingno as msno



In [3]:
df = pd.read_csv("https://docs.google.com/spreadsheets/d/19cEnOclBkX3qLrnlJtomQdXlev5gGr_e4fymYTr0giI/export?format=csv")

df.head()

/tmp/ipykernel_1992/3654750756.py:1: DtypeWarning: Columns (69,70) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("https://docs.google.com/spreadsheets/d/19cEnOclBkX3qLrnlJtomQdXlev5gGr_e4fymYTr0giI/export?format=csv")


,fight_outcome,outcome,event_date,event_name,event_location,fighter_a_name,fighter_b_name,weight_class,round,time,...,fighter_b_round1_opponent_control_time,fighter_b_round1_opponent_reversals,fighter_b_round1_opponent_sig_strikes_landed,fighter_b_opponent_head_strikes_landed,fighter_b_opponent_body_strikes_landed,fighter_b_opponent_leg_strikes_landed,fighter_b_opponent_distance_strikes_landed,fighter_b_opponent_clinch_strikes_landed,fighter_b_opponent_ground_strikes_landed,fight_minutes
0,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",ROYCE GRACIE,PATRICK SMITH,Open Weight,1.0,1:17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",ROYCE GRACIE,REMCO PARDOEL,Open Weight,1.0,1:31,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",ROYCE GRACIE,JASON DELUCIA,Open Weight,1.0,1:07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",PATRICK SMITH,SCOTT MORRIS,Open Weight,1.0,0:30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,red_win,1,1994-03-11,UFC 2: No Way Out,"Denver, Colorado, USA",JOHNNY RHODES,FRED ETTISH,Open Weight,1.0,3:07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df.shape

(8584, 124)

In [5]:
for i, column in enumerate(df.columns, start =1 ):
  print(f"{i}. {column}")

1. fight_outcome
2. outcome
3. event_date
4. event_name
5. event_location
6. fighter_a_name
7. fighter_b_name
8. weight_class
9. round
10. time
11. time_format
12. bout_type
13. referee
14. bonus
15. details
16. fighter_a_age
17. fighter_b_age
18. age_diff
19. fighter_a_reach
20. fighter_b_reach
21. reach_diff
22. fighter_a_height
23. fighter_b_height
24. fighter_a_weight
25. fighter_b_weight
26. fighter_a_sig_strikes_landed
27. fighter_b_sig_strikes_landed
28. fighter_a_sig_strikes_attempted
29. fighter_b_sig_strikes_attempted
30. fighter_a_total_strikes_landed
31. fighter_b_total_strikes_landed
32. fighter_a_total_strikes_attempted
33. fighter_b_total_strikes_attempted
34. fighter_a_body_strikes_landed
35. fighter_b_body_strikes_landed
36. fighter_a_body_strikes_attempted
37. fighter_b_body_strikes_attempted
38. fighter_a_clinch_strikes_landed
39. fighter_b_clinch_strikes_landed
40. fighter_a_clinch_strikes_attempted
41. fighter_b_clinch_strikes_attempted
42. fighter_a_distance_strik

In [6]:
null_counts = df.isnull().sum()

to_drop = null_counts[null_counts > 8000].index.to_list()


for dr in to_drop:
  df.drop(dr, inplace=True, axis=1)







In [8]:
df.columns

Index(['fight_outcome', 'outcome', 'event_date', 'event_name',
       'event_location', 'fighter_a_name', 'fighter_b_name', 'weight_class',
       'round', 'time', 'time_format', 'bout_type', 'referee', 'bonus',
       'details', 'fighter_a_age', 'fighter_b_age', 'age_diff',
       'fighter_a_reach', 'fighter_b_reach', 'reach_diff', 'fighter_a_height',
       'fighter_b_height', 'fighter_a_weight', 'fighter_b_weight',
       'fighter_a_sig_strikes_landed', 'fighter_b_sig_strikes_landed',
       'fighter_a_sig_strikes_attempted', 'fighter_b_sig_strikes_attempted',
       'fighter_a_total_strikes_landed', 'fighter_b_total_strikes_landed',
       'fighter_a_total_strikes_attempted',
       'fighter_b_total_strikes_attempted', 'fighter_a_body_strikes_landed',
       'fighter_b_body_strikes_landed', 'fighter_a_body_strikes_attempted',
       'fighter_b_body_strikes_attempted', 'fighter_a_clinch_strikes_landed',
       'fighter_b_clinch_strikes_landed', 'fighter_a_clinch_strikes_attempted',


In [7]:
fighters = pd.concat([df['fighter_a_name'], df['fighter_b_name']])


fighters.nunique()

2936

In [12]:
document_columns  = [
    "event_name",
    'event_date','event_location', 'fighter_a_name','weight_class',
 'fighter_b_name',  "fight_outcome",
    "method",    "round",
    "time",

    "bonus",
    "details",

    "fighter_a_sig_strikes_landed",
    "fighter_b_sig_strikes_landed",

    "fighter_a_total_strikes_landed",
    "fighter_b_total_strikes_landed",

    "fighter_a_knockdowns",
    "fighter_b_knockdowns",

    "fighter_a_takedowns_landed",
    "fighter_b_takedowns_landed",

    "fighter_a_submission_attempts",
    "fighter_b_submission_attempts"


]

filtered_df = df[document_columns]






,event_name,event_date,event_location,fighter_a_name,weight_class,fighter_b_name,fight_outcome,method,round,time,...,fighter_a_sig_strikes_landed,fighter_b_sig_strikes_landed,fighter_a_total_strikes_landed,fighter_b_total_strikes_landed,fighter_a_knockdowns,fighter_b_knockdowns,fighter_a_takedowns_landed,fighter_b_takedowns_landed,fighter_a_submission_attempts,fighter_b_submission_attempts
0,UFC 2: No Way Out,1994-03-11,"Denver, Colorado, USA",ROYCE GRACIE,Open Weight,PATRICK SMITH,red_win,KO/TKO,1.0,1:17,...,4,1,11.0,2.0,0,0,1,0,0,0
1,UFC 2: No Way Out,1994-03-11,"Denver, Colorado, USA",ROYCE GRACIE,Open Weight,REMCO PARDOEL,red_win,Submission,1.0,1:31,...,0,0,0.0,0.0,0,0,1,0,1,0
2,UFC 2: No Way Out,1994-03-11,"Denver, Colorado, USA",ROYCE GRACIE,Open Weight,JASON DELUCIA,red_win,Submission,1.0,1:07,...,0,0,0.0,0.0,0,0,0,1,1,0
3,UFC 2: No Way Out,1994-03-11,"Denver, Colorado, USA",PATRICK SMITH,Open Weight,SCOTT MORRIS,red_win,KO/TKO,1.0,0:30,...,13,0,19.0,0.0,0,0,0,0,0,0
4,UFC 2: No Way Out,1994-03-11,"Denver, Colorado, USA",JOHNNY RHODES,Open Weight,FRED ETTISH,red_win,Submission,1.0,3:07,...,13,4,21.0,7.0,1,0,0,0,1,0


In [13]:
df.to_csv("data/ufc_fights_clean.csv", index=False)